# HW3 — Robot Manipulation × Semantic Robot Knowledge

NYCU Physical AI / TAICA — Semantic Knowledge Graph + UR5 Kinematics + FSM Manipulation

## Learning Objectives

- **Grounding with Standard Robotics Ontologies**: Represent robot specifications and kinematic execution results as STRUCTURED RDF that reuses existing standard vocabularies — IEEE 1872 CORA robots, SOMA joints/joint-states/poses, IEEE 1872 POS positions, QUDT units — instead of ad-hoc strings. Symbol grounding turns execution numbers into machine-readable knowledge.
- **SHACL Validation of Execution Semantics**: Write SHACL shapes that validate the grounded execution data — discovering problem states of the IK/FK process (arm out of range, no convergence, joint-limit violation) directly from the numbers; your validation result on a TA-provided 30-record execution dataset must match the TA answer key exactly.
- **Forward Kinematics (FK)**: Understand and implement forward kinematics for a 6-DoF robot arm using D-H parameters, including end-effector pose and Jacobian computation.
- **Inverse Kinematics (IK)**: Understand and implement iterative inverse kinematics using Jacobian-based methods to compute joint configurations for a target end-effector pose.
- **Application of IK/FK in Robot Manipulation**: Understand how kinematics is integrated into a complete manipulation pipeline — a deterministic **Finite State Machine** drives the UR5 through pick-and-place episodes, solving every motion with YOUR IK and cross-checking YOUR FK against the simulator at every state transition.

## Overview

| Task | Content | Files you edit | Verification command | Points |
|---|---|---|---|---|
| 1 | Semantic Grounding (REUSE) + SHACL Validation | 2 functions in `semantic/ground_execution.py` + `semantic/shapes.ttl` | `pixi run task1` | 30 |
| 2 | Forward Kinematics | `fk.py` → `your_fk()` | `pixi run fk` | 20 |
| 3 | Inverse Kinematics | `ik.py` → `your_ik()` | `pixi run ik` | 40 |
| 4 | FSM Manipulation Pipeline | (integration, no code change) | `pixi run fsm` | 10 |

**Why the semantic layer comes first**: in Task 1 you build the knowledge representation and its validation rules BEFORE implementing any kinematics — the execution data is generated by the **TA reference solvers**, so Task 1 needs no FK/IK code. You therefore know, up front, exactly **which data classes and forms will be checked and what the expected results are**. And the connection is automatic: every run of `pixi run fk` / `pixi run ik` / `pixi run fsm` feeds its execution records back through **your** grounding functions and **your** `shapes.ttl` (via the TA-provided `semantic/diagnose.py`) and prints the SHACL problem flags right under the score — so Tasks 2–4 get **immediate semantic feedback** from the layer you built in Task 1. You can also re-run the full pipeline with `--own` at any time.

## Step 0. Environment Setup

As in the previous assignments, the environment is managed with [Pixi](https://pixi.sh) — every dependency (Python 3.7, PyBullet, NumPy, SciPy, **and OpenJDK 11**) is installed by Pixi from the shipped `pixi.toml`/`pixi.lock`, so no conda environment and no system Java are needed. The only thing Pixi cannot install is Pixi itself (plus a Linux OS; no GPU is used, no dataset or model is downloaded).

**Step 0-1.** Install Pixi once per machine, then create the environment:

In [ ]:
%%bash
curl -fsSL https://pixi.sh/install.sh | bash   # once per machine
# cd hw3   (not needed: this notebook already runs from hw3/)
pixi install        # resolves python=3.7, pybullet, numpy, scipy, openjdk=11

**Step 0-2.** Run everything through `pixi run` — predefined tasks or ad-hoc commands:

In [ ]:
%%bash
pixi run task1      # = bash semantic/run_task1.sh
pixi run fk         # = python fk.py
pixi run ik         # = python ik.py
pixi run fsm        # = python fsm_task.py
pixi run python fk.py -g -vp    # any ad-hoc command also works

The first run of Task 1 automatically downloads Apache Jena 4.10.0 (~30 MB; its `shacl` CLI performs the validation). It is cached under `semantic/.cache/` and works offline afterwards; network access is needed only for `pixi install` and that one download.

## Task 1. Semantic Grounding (REUSE) + SHACL Validation (30 points)

Turn the *numbers* of an FK/IK **execution process** into *semantics*, then let the semantic layer find what went wrong. By default the execution data comes from the **TA reference solvers** (ground-truth FK poses + the simulator IK, clipped to the course joint limits), so this task is done FIRST — before any FK/IK implementation. You implement two things — **grounding** and **SHACL**:

```
 TA reference solvers      (1) grounding
 (later: --own = your  ──>  ground_execution.py  ──>  data.ttl
  FK/IK execution)          (2 TODO functions)
                                                  (2) SHACL
                            shapes.ttl (3 TODO shapes)  vs  data.ttl            -> validation.ttl
                            shapes.ttl                  vs  ta-faulty-execution.ttl
                                                            (TA dataset, 30 cases) -> probe-validation.ttl
```

**Step 1-1. Understand the REUSEd vocabulary.** Open [semantic/ontology/hw3-ontology.ttl](semantic/ontology/hw3-ontology.ttl):
- Design rule: **reuse existing standard vocabularies wherever possible** — robots are `cora:Robot` (IEEE 1872 CORA), joints are `soma:RevoluteJoint`, joint states are `soma:JointState`, poses are `soma:6DPose` composed of an IEEE 1872 `pos:Position` + quaternion component, units are QUDT IRIs (`unit:M` / `unit:RAD`); `hw3:` only mints what no standard covers (kinematic computations, D-H parameters, statuses, distances)
- **Structured representation, no JSON strings**: numeric values are typed `xsd:double` literals on structured nodes. `semantic/common.py` provides the serializers (`pose_to_ttl` / `joint_config_to_ttl`) — use them

**Step 1-2. GROUNDING — implement the ontology-API functions.** Open [semantic/ground_execution.py](semantic/ground_execution.py):

| Function | Grounds | Status |
|---|---|---|
| `fk_computation_to_triples()` | one FK execution: input config, EE pose, pose/Jacobian errors | **worked example** — read it first |
| `robot_spec_to_triples()` | robot spec: D-H params + joint limits for all 6 joints | **TODO 1** |
| `ik_computation_to_triples()` | one IK execution: status, residual, `hw3:hasTargetDistance`, structured joint configuration | **TODO 2** |

The script grounds 3 FK executions (course ground truth) and 3 IK executions (reference solver on the fixed near / mid / far targets) and writes everything into **`output/data.ttl`** — an execution process as a semantic graph.

**Step 1-3. SHACL — write the validation shapes.** Open [semantic/shapes.ttl](semantic/shapes.ttl): the FK-accuracy shape (`hasPoseError <= 0.005` else `FK_INACCURATE`) is the worked example; you add **three** shapes — `hasTargetDistance <= 0.90` else **`ARM_OUT_OF_RANGE`** (arm beyond the UR5 workspace), `hasResidual <= 0.02` else **`NO_CONVERGENCE`**, and a **SHACL-SPARQL joint-limit check**: every `soma:JointState` angle must lie within its own joint's `[hasJointLowerLimit, hasJointUpperLimit]` (limits are inclusive), else **`JOINT_LIMIT_VIOLATION`** — SHACL Core cannot compare values from two different nodes, so this one uses `sh:sparql` (the `sh:prefixes` declaration is provided in the file). Message prefixes are graded, keep them exact. Two validations run:
- your shapes vs **your** `data.ttl` → `output/validation.ttl` — the reference execution flags exactly `target_mid` / `target_far` (they really are out of range) and leaves `target_near` clean
- your shapes vs **[semantic/ta-faulty-execution.ttl](semantic/ta-faulty-execution.ttl)** — a TA-provided dataset of **30 execution records** — IK, FK, and joint-state records, good/bad mixed, *including values exactly at the thresholds and a joint angle exactly at its limit*. The grader compares your validation result against the TA answer key **[semantic/ta-answer-key.json](semantic/ta-answer-key.json)** case by case — each record's flags must match the answer **exactly** (no false positives, no false negatives) to count as correct

Design note: SHACL validates the **numbers** (thresholds on distances / residuals / errors / limits) directly from the grounded data — the numeric checks stay in the data layer, the constraints live declaratively in shapes.

**Step 1-4. Run and score with one command.**

In [ ]:
%%bash
GROUP_ID="<your-group-id>"   # set your group id

pixi run task1 --group "$GROUP_ID"

The script runs 5 steps: toolchain → Jena → grounding (`data.ttl`) → SHACL (three validation reports) → scoring. Expected output:

```
  [S1] grounding structure (0 violations) OK  (+12)
  [S2] target_mid flagged ARM_OUT_OF_RANGE .... OK  (+2)
  [S2] target_far flagged ARM_OUT_OF_RANGE .... OK  (+2)
  [S2] target_near clean (no problem flags) ... OK  (+2)
  [S2] no JOINT_LIMIT_VIOLATION ............... OK  (+2)
  [S3] TA dataset vs answer key: faulty 14/14, clean 16/16 OK  (+10)
  Your Task 1 Score : 30.0 / 30.0
```

Grading uses **[semantic/ta-shapes-full.ttl](semantic/ta-shapes-full.ttl)** — the TA's complete SHACL suite: `STRUCTURE:*` shapes check your grounding (types, required properties, `xsd:double` typing, exactly 6 `soma:JointState` per configuration), problem shapes re-discover ARM_OUT_OF_RANGE / NO_CONVERGENCE from your numbers, and a SHACL-SPARQL shape cross-checks every joint angle against its joint's limits (`JOINT_LIMIT_VIOLATION`).

**Step 1-5. (Later, after Tasks 2–3.)** Re-run the pipeline on **your own** solvers and watch the semantic layer validate your implementation:

In [ ]:
%%bash
GROUP_ID="<your-group-id>"   # set your group id

pixi run task1 --group "$GROUP_ID" --own

**Task 1 FAQ**
- Not using Pixi? → any Python 3.7 with pybullet/numpy/scipy plus a system JDK 11+ works: `PYTHON=<your-python> bash semantic/run_task1.sh`
- Jena download fails → manually extract apache-jena-4.10.0 and `export JENA_HOME=<path>` before running
- S1 reports STRUCTURE violations → open `output/ta-validation.ttl`; each `sh:resultMessage` tells you which property/typing is missing (a bare `0.0892` instead of `"0.0892"^^xsd:double` is the classic one)
- S3 mismatches → the grader prints `expected [...] , got [...]` per wrong case. Your `sh:message` must START with `ARM_OUT_OF_RANGE:` / `NO_CONVERGENCE:` / `JOINT_LIMIT_VIOLATION:` (prefix-matched); numeric thresholds must use `sh:maxInclusive` (records exactly at 0.90 / 0.02 / 0.005 are *conforming*), and the joint-limit SPARQL `FILTER` must use strict `<` / `>` (an angle exactly at a limit is legal)

### Task 1 dataflow and mathematical model

```mermaid
flowchart TB
    R["TA reference<br/>FK/IK executions"] --> G["Semantic grounding<br/>ground_execution.py"]
    G --> D["output/data.ttl"]
    D --> V["SHACL validation"]
    S["student<br/>shapes.ttl"] --> V
    P["TA 30-record<br/>probe dataset"] --> V
    V --> O["validation reports"]
```

Let an RDF data graph be a finite triple set

$$
G \subseteq \mathcal N \times \mathcal P \times (\mathcal N \cup \mathcal L).
$$

Use $C(x) \Leftrightarrow (x,\texttt{rdf:type},C)\in G$ and
$p(x,y) \Leftrightarrow (x,p,y)\in G$. Grounding is a serialization function

$$
\gamma:\mathcal E\rightarrow 2^G,\qquad
G_{\mathrm{exec}}=G_{\mathrm{ontology}}\cup
\bigcup_{e\in\mathcal E}\gamma(e).
$$

A structured joint configuration $c$ must satisfy

$$
\textsf{JointConfiguration}(c)\ \land\left|\{s:\textsf{hasJointState}(c,s)\}\right|=6,
$$

and each state must link to its joint and numeric position. Given shapes graph
$S$ and data graph $G$,

$$
\mathrm{conforms}(S,G)
\Leftrightarrow
\mathrm{Report}(S,G)=\varnothing.
$$

The numerical conformance conditions are:

| Formula | Meaning | Violation flag |
|---|---|---|
| $e_x\le 0.005$ for an FK computation | FK pose accuracy | `FK_INACCURATE` |
| $d_x\le 0.90$ for an IK computation | arm reach | `ARM_OUT_OF_RANGE` |
| $r_x\le 0.02$ for an IK computation | IK convergence | `NO_CONVERGENCE` |
| $\ell_j\le q_s\le u_j$ for a joint state | joint limits | `JOINT_LIMIT_VIOLATION` |

All inequalities are inclusive. Values exactly at a threshold or joint limit
conform. The joint-limit violation query therefore uses
`FILTER (?v < ?lo || ?v > ?hi)`.

## Task 2. Forward Kinematics (20 points)

**Step 2-1.** Open [fk.py](fk.py) and read the D-H parameter table in `get_ur5_DH_params()`. It follows this project's `ur5.urdf` and differs slightly from the official UR5 spec — always use this table.

**Step 2-2.** Implement `your_fk(DH_params, q, base_pos)`:
- Input: 6 sets of D-H parameters, 6 joint angles `q`, base position `base_pos`
- Output: `(pose_7d, jacobian)` — the 7d end-effector pose `[x, y, z, qx, qy, qz, qw]` and the 6×6 geometric Jacobian
- **Do NOT use any pybullet API**; do not touch the `adjustment` block at the end of the function

**Step 2-3.** Verify (easy / medium / hard test cases; grading uses hidden ta1/ta2 cases):

In [ ]:
%%bash
pixi run fk                       # add -g for GUI, -vp to visualize poses:
pixi run python fk.py -g -vp

Expected output (Error Count must be 0 for every test file):

```
============================ Task 2 : Forward Kinematic ============================
- Testcase file : fk_test_case_easy.json
- Your Score Of Forward Kinematic : ... Error Count :    0 /  ...
- Your Score Of Jacobian Matrix   : ... Error Count :    0 /  ...
------------- Semantic diagnosis (your Task 1 layer) -------------
  [Task 2 FK] all 3 sampled records conform to your shapes (no problem flags)
```

After the score, the run feeds failing cases (plus one clean sample per file) through **your Task 1 grounding + shapes** — inaccurate cases show up as `FK_INACCURATE` / `JOINT_LIMIT_VIOLATION` flags. If Task 1 is unfinished the diagnosis just prints a skip notice; the score is unaffected.

## Task 3. Inverse Kinematics (40 points)

**Step 3-1.** Open [ik.py](ik.py) and implement `your_ik(robot_id, new_pose, base_pos, ...)`:
- Use your `your_fk` and its Jacobian in an iterative method (e.g., damped least squares / pseudo-inverse)
- Be careful when computing delta x; tune hyper-parameters such as the step rate. Grading calls your function with **default arguments only**, so make your defaults the best ones
- `pybullet_ik()` in the same file is a reference you may compare behavior against, but your implementation must not call pybullet IK APIs

**Step 3-2.** Verify:

In [ ]:
%%bash
pixi run ik

Expected output: Error Count 0 on all three test files, Mean Error in the 1e-3 range, and the semantic diagnosis reporting conforming records:

```
------------- Semantic diagnosis (your Task 1 layer) -------------
  [Task 3 IK] all 3 sampled records conform to your shapes (no problem flags)
```

Failing IK cases are grounded through your Task 1 functions and flagged by your shapes as `NO_CONVERGENCE` / `ARM_OUT_OF_RANGE` / `JOINT_LIMIT_VIOLATION` — the same vocabulary you defined before implementing anything.

**Step 3-3.** Now that both solvers are yours, revisit Task 1: `pixi run task1 --group <id> --own` grounds and validates **your** execution.

## Task 4. FSM Manipulation Pipeline (10 points)

Your IK/FK now drive a complete manipulation pipeline — no learned model, no
downloads: a deterministic **Finite State Machine** ([fsm_task.py](fsm_task.py),
TA-provided, no code change) runs 10 seeded pick-and-place episodes:

```mermaid
flowchart TB
    A[MOVE_TO_PREPICK] --> B[DESCEND_TO_PICK] --> C[GRASP] --> D[LIFT]
    D --> E[MOVE_TO_PREPLACE] --> F[DESCEND_TO_PLACE] --> G[RELEASE] --> H[RETREAT]
    H --> I[VERIFY]
    I -->|pass| J[SUCCESS]
    I -->|fail| K[FAILURE]
```

Every motion state performs **two validations**:
- **IK validation** — the target pose is solved by `your_ik`, executed with
  position control, and the achieved end-effector position must land within
  3 cm of the command (`IK_MISS` otherwise)
- **FK validation** — `your_fk` is evaluated on the MEASURED joint angles and
  must agree with the simulator's end-effector position within 1 cm
  (`FK_MISMATCH` otherwise) — your kinematic model is checked against physics
  at every state transition

`DESCEND_TO_PICK` steps downward until the suction gripper reports contact,
`GRASP`/`RELEASE` toggle the suction constraint, and `VERIFY` requires the
block to rest within 6 cm of the goal marker.

**Run** (10 seeded episodes; add `-g` to watch in the GUI):

In [ ]:
%%bash
pixi run fsm
pixi run python fsm_task.py -g    # PyBullet window: watch the FSM drive the arm

Expected output:

```
============================ Task 4 : FSM Manipulation Pipeline ============================
FSM: MOVE_TO_PREPICK -> DESCEND_TO_PICK -> GRASP -> ... -> VERIFY
Episode  1/10 | SUCCESS  (place error 0.000 m)
...
Episode 10/10 | SUCCESS  (place error 0.000 m)
- Your Total Score : 10.000 / 10.000
------------- Semantic diagnosis (your Task 1 layer) -------------
  [Task 4 FSM] all 1 sampled records conform to your shapes (no problem flags)
```

A failure names the state and the reason (`IK_MISS in MOVE_TO_PREPICK ...`,
`FK_MISMATCH in LIFT ...`, `NO_CONTACT`, `PLACE_MISS`) — the FSM localizes
WHERE in the pipeline your kinematics broke, and the failing motions are
additionally grounded and SHACL-validated by **your Task 1 layer**, so each
failure also carries its semantic flags (`NO_CONVERGENCE`, `ARM_OUT_OF_RANGE`,
`JOINT_LIMIT_VIOLATION`, `FK_INACCURATE`).

## Grading

| Item | Points |
|---|---|
| Task 1: S1 grounding structure 12 + S2 problem detection 8 + S3 SHACL vs TA dataset & answer key 10 | 30 |
| Task 2: FK correctness (FK 10 + Jacobian 10, TA test cases) | 20 |
| Task 3: IK correctness (TA test cases, default arguments only) | 40 |
| Task 4: FSM pick-and-place success over 10 seeded episodes | 10 |
| **Total** | **100** |

## Submission

1. `semantic/ground_execution.py` (with both TODO functions completed)
2. `semantic/shapes.ttl` (with the three problem shapes completed)
3. `fk.py` (with `your_fk` completed)
4. `ik.py` (with `your_ik` completed)
5. `<group-id>_hw3_report.pdf`, following the required four-section format below

> Note: `semantic/output/`, `semantic/store/`, `semantic/.cache/`, and `.pixi/` are auto-generated — do not submit them.

## Required PDF Report Format

Submit `<group-id>_hw3_report.pdf`. The cover must list the course, homework
title, group ID, every member's student ID and name, and the submission date.
Use exactly these four numbered top-level sections:

### 1. Semantic Grounding and SHACL Validation

1. **Grounding design:** explain both grounding TODO functions, reused
   CORA/SOMA/POS/QUDT terms, structured nodes, and `xsd:double` values.
2. **SHACL constraints:** give each target class, path, inclusive threshold,
   flag prefix, and explain why joint limits require SHACL-SPARQL.
3. **Evidence:** include a readable screenshot of the complete
   `run_task1.sh` output showing S1, four S2 checks, S3 counts, and total /30.
4. **Analysis:** interpret both `output/validation.ttl` and
   `output/probe-validation.ttl`; discuss mid/far/near, one boundary record,
   one joint-limit record, and any false positive/negative.

### 2. Forward Kinematics and Geometric Jacobian

1. **Method:** explain classic D-H transform composition, pose/quaternion,
   base position, and the protected adjustment.
2. **Jacobian:** derive
   $J_{v,i}=z_{i-1}\times(p_e-p_{i-1})$ and $J_{\omega,i}=z_{i-1}$.
3. **Evidence:** include the complete final `pixi run fk` output, all test
   files, both error counts, total /20, and semantic diagnosis.
4. **Analysis:** tabulate per-file FK/Jacobian scores and error counts. For a
   failure, report its norm, threshold, likely cause, and attempted fix.

### 3. Inverse Kinematics

1. **Method:** explain pose error, relative-rotation vector, damped least
   squares, warm start, stopping condition, and joint-limit clipping.
2. **Parameters:** report final default damping, step rate, maximum iterations,
   and stopping threshold, with the reason for each choice.
3. **Evidence:** include the complete final `pixi run ik` output, per-file
   mean/error count/score, total /40, and semantic diagnosis.
4. **Analysis:** tabulate results and explain at least one difficulty, the
   diagnostic evidence, and the final fix.

### 4. FSM Manipulation Pipeline

1. **Pipeline:** explain pick, grasp, transport, place, release, retreat, and
   verification; identify where student IK and FK are used.
2. **Evidence:** show all ten episode results, total /10, and semantic
   diagnosis. Use multiple screenshots if needed for legibility.
3. **Analysis:** report successes/failures and each failure's first state and
   exact reason (`IK_MISS`, `FK_MISMATCH`, `NO_CONTACT`,
   `GRASP_FAILED`, or `PLACE_MISS`).
4. **Conclusion:** explain end-to-end agreement among semantics, FK, IK, and
   FSM, then identify one limitation or concrete improvement.

Number and caption every figure/table, cite it from the prose, keep terminal
text readable, include units, and use selectable text for explanations.
Screenshots alone are not a report. Each screenshot must show the command and
results from the final submitted implementation.

## Reference

- https://automaticaddison.com/the-ultimate-guide-to-jacobian-matrices-for-robotics/
- Apache Jena / TDB2: https://jena.apache.org/documentation/tdb2/